# Lakehouse Federation

## Connection to the Neon database
Password was set up using Databricks CLI.

In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS neon_pg TYPE postgresql
OPTIONS (
    host    'ep-nameless-brook-b26w7gmr-pooler.c-6.eu-central-1.aws.neon.tech',
    port    '5432',
    user    'neondb_owner',
    password secret('neon', 'pg_password'),
    trustServerCertificate 'true'
);

## Creating a foreign catalog to mirror the Neon database into Unity Catalog

In [0]:
%sql
CREATE FOREIGN CATALOG IF NOT EXISTS neon USING CONNECTION neon_pg
OPTIONS (database 'neondb');

## Sanity check

In [0]:
%sql
SELECT * FROM neon.public.dc_dim ORDER BY it_power_mw DESC;

## Joining the external table to the Delta Table
The dc_dim table is also used in the gold layer of the Entsoe project as an additional dimension. Here, the join is done for the purposes of the Lab.

In [0]:
# Configuration
from pyspark.sql import functions as F, Window
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
CATALOG       = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")


In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly LIMIT 10"))

In [0]:
# Joined table with the latest consumption data and data center metadata
joined_table = spark.sql(f"""
SELECT
cons.site_id,
cons.bidding_zone,
cons.date,
cons.hour,
cons.avg_consumption_kwh AS avg_consumption_kwh,
CASE
WHEN cons.avg_consumption_kwh IS NOT NULL THEN true
ELSE FALSE
END AS is_energy_consumed,
cons.avg_power_kw AS avg_power_kw,
cons.avg_pue AS avg_pue,
cons.cost_per_hour AS cost_per_hour,
dc.operator,
dc.city,
dc.longitude,
dc.latitude,
dc.it_power_mw,
dc.power_density_kw_m2
FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly AS cons
LEFT JOIN neon.public.dc_dim AS dc
ON cons.site_id = dc.dc_id
QUALIFY ROW_NUMBER() OVER (PARTITION BY cons.site_id ORDER BY cons.date DESC, cons.hour DESC) = 1;""")

display(joined_table)


## Change Data Capture

In [0]:
spark.sql(f"""CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.cdc_table 
          TBLPROPERTIES(delta.enableChangeDataFeed = true) 
          AS SELECT * FROM neon.public.dc_dim""")

display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{GOLD_SCHEMA}.cdc_table"))

The initial load version is 0. Versions 1 & 2 are the updates of the entire table in Neon. Now updating the table and checking the history again.


In [0]:
# Updating the it_power_mw and related columns for the German data center
spark.sql(f"""UPDATE {CATALOG}.{GOLD_SCHEMA}.cdc_table
              SET it_power_mw = 60.0,
                  power_density_kw_m2 = ROUND(60.0 * 1000 / tech_area_m2, 2),
                  pue = 1.15
              WHERE dc_id = 'DC-DE-01'""")
# Updating the tier of the Polish data center
spark.sql(f"""UPDATE {CATALOG}.{GOLD_SCHEMA}.cdc_table
              SET tier = 4 WHERE dc_id = 'DC-PL-01'""")

# Adding a new row for the Irish data center
spark.sql(f"""INSERT INTO {CATALOG}.{GOLD_SCHEMA}.cdc_table VALUES
              ('DC-IE-01','Dublin West','CelticEdge','IE','IE_SEM','Dublin',
               53.3990,-6.4460,15000,33.0,ROUND(33.0*1000/15000,2),4,1.19,2024,'liquid')""")

In [0]:
spark.sql(f"""CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.cdc_table 
          TBLPROPERTIES(delta.enableChangeDataFeed = true) 
          AS SELECT * FROM neon.public.dc_dim""")

display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{GOLD_SCHEMA}.cdc_table"))

Now, we can see singular updates from 3 to 7 performed in Databricks.

We can also see the details of changes applied to DC-DE-01:

In [0]:
display(spark.sql(f"""
          SELECT * FROM table_changes('{CATALOG}.{GOLD_SCHEMA}.cdc_table', 1)
          WHERE dc_id = 'DC-DE-01'
          ORDER BY _commit_timestamp"""))

## SCD2

Creating a new table with SCD2:

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{GOLD_SCHEMA}.dc_dim_scd2 AS
SELECT *,
  current_timestamp()     AS valid_from,
  CAST(NULL AS TIMESTAMP) AS valid_to,
  true                    AS is_current
FROM {CATALOG}.{GOLD_SCHEMA}.cdc_table
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dc_dim_scd2"))

In [0]:
# 1. Closing all old records for rows that have been updated or deleted
step1 = spark.sql(f"""
MERGE INTO {CATALOG}.{GOLD_SCHEMA}.dc_dim_scd2 t
USING (
  SELECT * FROM (
    SELECT *, row_number() OVER (PARTITION BY dc_id ORDER BY _commit_version DESC) rn
    FROM table_changes('{CATALOG}.{GOLD_SCHEMA}.cdc_table', 1)
    WHERE _change_type IN ('insert','update_postimage','delete')
  ) WHERE rn = 1
) s
ON t.dc_id = s.dc_id AND t.is_current = true
WHEN MATCHED THEN UPDATE SET is_current = false, valid_to = s._commit_timestamp
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dc_dim_scd2"))


In [0]:
# 2. Inserting new records for rows that have been inserted or updated

step2 = spark.sql(f"""
INSERT INTO {CATALOG}.{GOLD_SCHEMA}.dc_dim_scd2
SELECT
  dc_id, dc_name, operator, country_code, bidding_zone, city, latitude, longitude,
  tech_area_m2, it_power_mw, power_density_kw_m2, tier, pue, commissioned_year,
  cooling_type,
  _commit_timestamp AS valid_from, CAST(NULL AS TIMESTAMP) AS valid_to, true AS is_current
FROM (
  SELECT *, row_number() OVER (PARTITION BY dc_id ORDER BY _commit_version DESC) rn
  FROM table_changes('{CATALOG}.{GOLD_SCHEMA}.cdc_table', 1)
  WHERE _change_type IN ('insert','update_postimage','delete')
) x
WHERE rn = 1 AND _change_type IN ('insert','update_postimage')
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dc_dim_scd2"))